In [54]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/home/jovyan/kuratov/data/test_time_gd/')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
from kv_dataset_utils import generate_sequence, get_extra_chars, BASE_KV_ALPHABET

In [75]:
num_kv_pairs = 4
k_length = 2
v_length = 2
n_segments = 1
min_segment_len = 0
max_segment_len = 0

kv_vocab_size = 512

kv_alphabet = BASE_KV_ALPHABET + get_extra_chars(kv_vocab_size)

sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len, kv_alphabet)

sample


{'kv_pairs': ['!Ы😶:ü⑤!', '!ÌÂ:ü9!', '!Ⓑ⑦:😘⑮!', '!⑲°:⓴Й!'],
 'segment_ids_to_kv_ids': {0: [0, 1, 2, 3]},
 'context': '!⑲°:⓴Й!!Ⓑ⑦:😘⑮!!ÌÂ:ü9!!Ы😶:ü⑤!|',
 'query': '?!⑲°:',
 'input_sequence': '!⑲°:⓴Й!!Ⓑ⑦:😘⑮!!ÌÂ:ü9!!Ы😶:ü⑤!|?!⑲°:',
 'target': '⓴Й!|'}

In [76]:
print('KV pairs length:', (k_length + v_length + 3)*num_kv_pairs)
print('~Total length:', n_segments * (min_segment_len + max_segment_len)/2)

KV pairs length: 28
~Total length: 0.0


In [77]:
sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len, kv_alphabet)
sample

{'kv_pairs': ['!ЦU:Β▦!', '!😠😗:Ⓔ◅!', '!ì▾:ØP!', '!Д¦:😣ⓘ!'],
 'segment_ids_to_kv_ids': {0: [0, 1, 2, 3]},
 'context': '!Д¦:😣ⓘ!!ì▾:ØP!!😠😗:Ⓔ◅!!ЦU:Β▦!|',
 'query': '?!ì▾:',
 'input_sequence': '!Д¦:😣ⓘ!!ì▾:ØP!!😠😗:Ⓔ◅!!ЦU:Β▦!|?!ì▾:',
 'target': 'ØP!|'}

In [78]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm

num_samples = 1_000_000

data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                               min_segment_len, max_segment_len, kv_alphabet)
    data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]
data = Dataset.from_list(data)

num_samples = 5_000

valid_data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                               min_segment_len, max_segment_len, kv_alphabet)
    valid_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]
valid_data = Dataset.from_list(valid_data)

dataset = DatasetDict({'train': data, 'valid': valid_data})

  0%|          | 0/1000000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [00:00<00:00, 34647.98it/s]


In [79]:
dataset

DatasetDict({
    train: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 1000000
    })
    valid: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 5000
    })
})

In [80]:
if n_segments == 1 and min_segment_len == 0 and max_segment_len == 0:
    # no noise dataset
    dataset_name = f'N{num_kv_pairs}-K{k_length}V{v_length}-V{kv_vocab_size}_1M'
else:
    dataset_name = f'N{num_kv_pairs}-K{k_length}V{v_length}-S{n_segments}({min_segment_len}-{max_segment_len})_1M'
print(dataset_name)
dataset.save_to_disk(f'./data/{dataset_name}')

N4-K2V2-V512_1M


Saving the dataset (0/1 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

# copy task dataset

In [12]:
from kv_dataset_utils import generate_sequence

num_kv_pairs = 0
k_length = 4
v_length = 4
n_segments = 1
min_segment_len = 4
max_segment_len = 4

sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len)
sample

{'kv_pairs': [],
 'segment_ids_to_kv_ids': {0: []},
 'context': 'v42v|',
 'query': '?!:',
 'input_sequence': 'v42v|?!:',
 'target': ''}

In [13]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm

num_samples = 1_000_000

data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['context'],
    }]
data = Dataset.from_list(data)

num_samples = 5_000

valid_data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    valid_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['context'],
    }]
valid_data = Dataset.from_list(valid_data)

dataset = DatasetDict({'train': data, 'valid': valid_data})

100%|██████████| 5000/5000 [00:00<00:00, 170832.10it/s]


In [14]:
dataset_name = f'N{num_kv_pairs}-S{n_segments}({min_segment_len}-{max_segment_len})_1M'
dataset.save_to_disk(f'./data/{dataset_name}')

Saving the dataset (0/1 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [15]:
dataset = dataset.load_from_disk(f'./data/{dataset_name}')